<a href="https://colab.research.google.com/github/Yusuf-Sonmez/Exercises/blob/main/Tweet_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets

In [2]:
import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader,Dataset
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,confusion_matrix

In [3]:
dataset = load_dataset("winvoker/turkish-sentiment-analysis-dataset")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/76.1M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/440679 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/48965 [00:00<?, ? examples/s]

In [4]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

In [5]:
print(train_df.head())

                                                text     label         dataset
0  ürünü hepsiburadadan alalı 3 hafta oldu. orjin...  Positive  urun_yorumlari
1  ürünlerden çok memnunum, kesinlikle herkese ta...  Positive  urun_yorumlari
2      hızlı kargo, temiz alışveriş.teşekkür ederim.  Positive  urun_yorumlari
3               Çünkü aranan tapınak bu bölgededir .      Notr            wiki
4  bu telefonu başlıca alma nedenlerim ise elimde...  Positive  urun_yorumlari


In [6]:
print(train_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440679 entries, 0 to 440678
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   text     440679 non-null  object
 1   label    440679 non-null  object
 2   dataset  440679 non-null  object
dtypes: object(3)
memory usage: 10.1+ MB
None


In [7]:
df_pozitif = train_df[train_df['label']== 'Positive'].sample(n=15000,random_state=42)
df_notr = train_df[train_df['label']== 'Notr'].sample(n=15000,random_state=42)
df_negatif = train_df[train_df['label']== 'Negative'].sample(n=15000,random_state=42)

train_df_balanced = pd.concat([df_pozitif,df_notr,df_negatif]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Hazır! Veri Seti Boyutu:", len(train_df_balanced))

Hazır! Veri Seti Boyutu: 45000


In [8]:
print(train_df_balanced.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     45000 non-null  object
 1   label    45000 non-null  object
 2   dataset  45000 non-null  object
dtypes: object(3)
memory usage: 1.0+ MB
None


In [9]:
train_df_balanced.drop(columns=['dataset'], inplace=True)
test_df.drop(columns=['dataset'], inplace=True)

In [10]:
model_name = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [11]:
ornek_tweet = train_df_balanced['text'].iloc[0]
print("1. Orijinal metin:\n", ornek_tweet)

tokens = tokenizer.tokenize(ornek_tweet)
print("\n2. Tokenler:\n", tokens)

token_ids = tokenizer.encode(ornek_tweet)
print("\n3. IDs:\n", token_ids)

1. Orijinal metin:
 berbat şuanda otelde tatildeyiz otel tesis hizmetler güzel fakat havuz olayı berbat kızımla havuza girmek istiyorum yaşı küçük olmaz diyorlar onun havuzunda yanında bile oturamıyorsunuz yani cocuk sizin sizde cocugun havuzuna giremiosun böylece ya siz yüzmüceksiniz ya o tatilden cok işkence oldu cocuklu kişilere aslaaaaa tavsiye etmiyorum cocuk parkıda cocuklara uyun deil kendi başlarına oynayabilecekleri gb deil ozaman cocuk parkının ne anlamı var yaşındaki oglumuzzda bayanlar havuzuna giremiyor erkekler havuzunda cocuk havuzu yok böle sacma sapan bi durum bilseydim gelmezdim paramızla rezil olduk

2. Tokenler:
 ['berbat', 'şuanda', 'otelde', 'tatil', '##dey', '##iz', 'otel', 'tesis', 'hizmetler', 'güzel', 'fakat', 'havuz', 'olayı', 'berbat', 'kızım', '##la', 'havuz', '##a', 'girmek', 'istiyorum', 'yaşı', 'küçük', 'olmaz', 'diyorlar', 'onun', 'havuzu', '##nda', 'yanında', 'bile', 'otur', '##amıyor', '##sunuz', 'yani', 'cocuk', 'sizin', 'sizde', 'co', '##cu', '##gun

In [12]:
ornek_cumleler = ["Harika bir ürün!", "Berbat bir otel, sakın ama sakın gitmeyin."]

batch_tokens = tokenizer(
    ornek_cumleler,
    padding=True,
    truncation=True,
    max_lenght=15,
    return_tensors="pt"
)

print(batch_tokens)

{'input_ids': tensor([[    2, 16050,  1996,  2782,     5,     3,     0,     0,     0,     0,
             0,     0,     0],
        [    2,  5253,  5817,  1996,  4589,    16,  9288,  2262,  9288, 21476,
          1009,    18,     3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [13]:
label_mapping = {'Negative':0,'Notr':1,'Positive':2}

train_df_balanced['label_num'] = train_df_balanced['label'].map(label_mapping)
test_df['label_num'] = test_df['label'].map(label_mapping)

In [14]:
class TweetDataset(Dataset):
  def __init__(self, metinler, etiketler, tokenizer, max_len):
    self.metinler = metinler.values
    self.etiketler = etiketler.values
    self.tokenizer = tokenizer
    self.max_len = max_len

  def __len__(self):
    return len(self.metinler)

  def __getitem__(self, item):
    metin = str(self.metinler[item])
    etiket = self.etiketler[item]

    encoding = self.tokenizer(
        metin,
        add_special_tokens=True,
        max_length=self.max_len,
        padding='max_length',
        truncation=True,
        return_tensors = 'pt'
    )

    return {
        'input_ids': encoding['input_ids'].flatten(),
        'attention_mask': encoding['attention_mask'].flatten(),
        'labels': torch.tensor(etiket, dtype=torch.long)
    }

max_kelime_uzunlugu = 128
ornek_dataset = TweetDataset(
    metinler = train_df_balanced['text'],
    etiketler = train_df_balanced['label_num'],
    tokenizer = tokenizer,
    max_len = max_kelime_uzunlugu
)


ilk_veri = ornek_dataset[0]
print("Input IDs Boyutu:", ilk_veri['input_ids'].shape)
print("Attention Mask Boyutu:", ilk_veri['attention_mask'].shape)
print("Hedef Etiket (Label):" , ilk_veri['labels'].item())

Input IDs Boyutu: torch.Size([128])
Attention Mask Boyutu: torch.Size([128])
Hedef Etiket (Label): 0


In [15]:
max_kelime_uzunlugu = 128
train_dataset = TweetDataset(
    metinler = train_df_balanced['text'],
    etiketler = train_df_balanced['label_num'],
    tokenizer = tokenizer,
    max_len = max_kelime_uzunlugu

  )
test_dataset = TweetDataset(
    metinler = test_df['text'],
    etiketler = test_df['label_num'],
    tokenizer = tokenizer,
    max_len = max_kelime_uzunlugu
)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Model İndiriliyor...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

print("Eğitim seti paket (batch) sayısı:", len(train_loader))

Model İndiriliyor...


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Eğitim seti paket (batch) sayısı: 2813


In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Kullanılan Cihaz:", device)

model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

Kullanılan Cihaz: cuda


In [17]:
from tqdm import tqdm
epochs = 2

for epoch in range(epochs):
  model.train()
  total_loss = 0
  loop = tqdm(train_loader, leave=True)

  for batch in loop:

    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()

    outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    total_loss += loss.item()

    loss.backward()
    optimizer.step()

    loop.set_postfix(loss=loss.item())

  ortalama_loss = total_loss / len(train_loader)
  print(f"Epoch {epoch+1} Bitti! Ortalama Loss: {ortalama_loss:.4f}")

100%|██████████| 2813/2813 [17:13<00:00,  2.72it/s, loss=0.844]


Epoch 1 Bitti! Ortalama Loss: 0.1798


100%|██████████| 2813/2813 [17:23<00:00,  2.70it/s, loss=0.0258]

Epoch 2 Bitti! Ortalama Loss: 0.1067


In [20]:
from sklearn.metrics import classification_report

model.eval()

gercek_degerler = []
tahminler = []

print("Test verisi üzerinde değerlendirme başlıyor...")

with torch.no_grad():
  loop_test = tqdm(test_loader, leave=True)
  for batch in loop_test:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    outputs = model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits

    tahmin_edilen_sinif = torch.argmax(logits, dim=1)
    gercek_deger = labels.cpu().numpy()

    tahminler.extend(tahmin_edilen_sinif)
    gercek_degerler.extend(gercek_deger)

print("\n--- Sınıflandırma Raporu ---")
hedef_isimleri = ['Negative (0)', 'Notr (1)', 'Positive (2)']
print(classification_report(gercek_degerler, tahminler, target_names=hedef_isimleri))

print("\n--- Karmaşıklık Matrisi (Confusion Matrix) ---")
print(confusion_matrix(gercek_degerler, tahminler))


Test verisi üzerinde değerlendirme başlıyor...


100%|██████████| 3061/3061 [06:44<00:00,  7.57it/s]



--- Sınıflandırma Raporu ---


TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [22]:
import torch
from sklearn.metrics import classification_report, confusion_matrix

gercek_degerler_kurtarilmis = []
for g in gercek_degerler:
    if torch.is_tensor(g):
        # Tensörün içindeki o tekil saf sayıyı (.item()) cımbızla çekip listeye ekliyoruz
        gercek_degerler_kurtarilmis.append(g.item())
    else:
        gercek_degerler_kurtarilmis.append(g)

tahminler_kurtarilmis = []
for t in tahminler:
    if torch.is_tensor(t):
        # Aynı işlemi tahminler için de yapıyoruz
        tahminler_kurtarilmis.append(t.item())
    else:
        tahminler_kurtarilmis.append(t)

# 3. Raporu nihayet çalıştırıyoruz!
hedef_isimleri = ['Negative (0)', 'Notr (1)', 'Positive (2)']

print("\n--- Sınıflandırma Raporu ---")
print(classification_report(gercek_degerler_kurtarilmis, tahminler_kurtarilmis, target_names=hedef_isimleri))

print("\n--- Karmaşıklık Matrisi (Confusion Matrix) ---")
print(confusion_matrix(gercek_degerler_kurtarilmis, tahminler_kurtarilmis))


--- Sınıflandırma Raporu ---
              precision    recall  f1-score   support

Negative (0)       0.79      0.91      0.84      5656
    Notr (1)       1.00      1.00      1.00     17092
Positive (2)       0.98      0.95      0.96     26217

    accuracy                           0.96     48965
   macro avg       0.92      0.95      0.93     48965
weighted avg       0.96      0.96      0.96     48965


--- Karmaşıklık Matrisi (Confusion Matrix) ---
[[ 5142    13   501]
 [    7 17065    20]
 [ 1386    49 24782]]


In [23]:
import os

# Kaydedilecek klasörün adı
kayit_klasoru = "./benim_turkce_duygu_modelim"

# Klasör yoksa oluştur
if not os.path.exists(kayit_klasoru):
    os.makedirs(kayit_klasoru)

# Modeli ve Tokenizer'ı bu klasöre kaydediyoruz
model.save_pretrained(kayit_klasoru)
tokenizer.save_pretrained(kayit_klasoru)

print(f"Harika! Model ve Tokenizer başarıyla '{kayit_klasoru}' klasörüne kaydedildi.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Harika! Model ve Tokenizer başarıyla './benim_turkce_duygu_modelim' klasörüne kaydedildi.


In [26]:
import gradio as gr

kayit_klasoru = "./benim_turkce_duygu_modelim"

yerel_model = AutoModelForSequenceClassification.from_pretrained(kayit_klasoru)
yerel_tokenizer = AutoTokenizer.from_pretrained(kayit_klasoru)

yerel_model.eval()

def duygu_analizi_yap(metin):
  inputs = yerel_tokenizer(metin, return_tensors="pt", truncation=True, padding=True, max_length=128)
  with torch.no_grad():
    outputs = yerel_model(**inputs)
    logits = outputs.logits

    tahmin_id = torch.argmax(logits, dim=1).item()

  etiketler = {
      0: "Negatif",
      1: "Notr",
      2: "Pozitif"
  }

  return etiketler[tahmin_id]

arayuz = gr.Interface(
    fn = duygu_analizi_yap,
    inputs = gr.Textbox(lines=3, placeholder="Tweet giriniz ..."),
    outputs = gr.Label(label="Yapay Zeka Tahmini"),
    title = "Türkçe Tweet Duygu Analizi",
    description = "BERTurk modeli fine-tune edilerek geliştirilmiştir. Test etmek için bir şeyler yazın!"
)

arayuz.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bd887db826aa1b06d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
